In [ ]:
import pandas as pd
import openpyxl
import numpy as np
import re

# =====================================================
# LOAD INPUT FILE
# =====================================================
df = pd.read_excel("April - test1_or.xlsx")

# =====================================================
# TEXT NORMALIZATION
# =====================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).lower().strip()

# =====================================================
# AFTERNOON CHECK
# =====================================================
def is_afternoon(window):
    if pd.isna(window):
        return False
    start = str(window).split("-")[0].strip()
    return ("PM" in start) and (not start.startswith("12"))

df["is_afternoon"] = df["pickup_window"].apply(is_afternoon)

# =====================================================
# ZONE CLASSIFICATION
# =====================================================
def classify_zone(place):
    p = clean_text(place)
    if "sentosa" in p:
        return "SENTOSA"
    if "garden" in p or "marina bay sands" in p:
        return "GARDENS_BY_THE_BAY"
    return "OTHER"

df["zone"] = df["To"].apply(classify_zone)

# =====================================================
# GROUP KEY LOGIC
# =====================================================
def build_group_key(row):
    trip = clean_text(row["trip_type"])

    if "pickup" not in trip:
        return row["pickup_window"]

    if not row["is_afternoon"]:
        return row["pickup_window"]

    if row["zone"] in ["SENTOSA", "GARDENS_BY_THE_BAY"]:
        return f"{row['pickup_window']} | {row['zone']}"

    return row["pickup_window"]

df["group_key"] = df.apply(build_group_key, axis=1)

# =====================================================
# BATCH ID ASSIGNMENT
# (kept consistent with your original intention: factorize group_key within pickup_window)
# =====================================================
def assign_batch_id(window_group):
    window_group = window_group.copy()
    trip = clean_text(window_group["trip_type"].iloc[0])
    prefix = "p" if "pickup" in trip else "r"
    codes = pd.factorize(window_group["group_key"])[0] + 1
    window_group["batch_id"] = [f"{prefix}{i}" for i in codes]
    return window_group

df = (
    df.groupby("pickup_window", group_keys=False, sort=True)
      .apply(assign_batch_id)
)

# =====================================================
# LOAD PARKING CONSTRAINT FILE
# =====================================================
parking_df = pd.read_excel("parking_constraint_lat_split_fin1.xlsx")

def clean_hotel_name(x):
    x = str(x).lower()
    x = x.replace("singapore", "")
    x = x.replace(",", "")
    x = " ".join(x.split())
    return x.strip()

parking_df["Hotel_names_clean"] = parking_df["Hotel_names"].apply(clean_hotel_name)
df["From_clean"] = df["From"].apply(clean_hotel_name)

# =====================================================
# MERGE PARKING STATUS
# =====================================================
df = df.merge(
    parking_df[["Hotel_names_clean", "Parking_status"]],
    left_on="From_clean",
    right_on="Hotel_names_clean",
    how="left"
)

# =====================================================
# INITIAL VEHICLE ALLOCATION
# =====================================================
def get_vehicle_allocation(row):
    trip = str(row["trip_type"]).lower().strip()

    if "return" in trip:
        return "default"

    if "pickup" in trip:
        status = str(row["Parking_status"]).lower().strip()
        if status == "coach_allowed":
            return "coach"
        else:
            return "comfi"

    return "default"

df["initial_vehicle_allocation"] = df.apply(get_vehicle_allocation, axis=1)

# =====================================================
# STEP 2 BATCHING (FIXED) + VEHICLE UNIQUE ID
# - Correct vehicle reuse per pickup window
# - Correct splitting feasibility vs remaining pool
# - Best-fit coach selection (smallest that fits)
# - Hotel-level pax aggregation within each batch
# - Bidirectional neighbor matching in distance table
# - NEW: assigns a unique coach ID like coach001 (23 seater)
# =====================================================
dist_df = pd.read_excel("hotel_distance_time_with_ids_fin1.xlsx")
dist_df["From_clean"] = dist_df["From"].apply(clean_hotel_name)
dist_df["To_clean"]   = dist_df["To"].apply(clean_hotel_name)

def extract_km(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower().replace("km", "").strip()
    try:
        return float(x)
    except:
        return np.nan

dist_df["distance_km"] = dist_df["distance_text"].apply(extract_km)

df["total_pack"] = df["adult"].fillna(0) + df["child"].fillna(0)

# -------- USER INPUT VEHICLES --------
print("\nEnter available coaches like:")
print("Example => 2x23, 1x42, 1x49, 1x55")

user_input = input("\nEnter coach availability: ")

# Build coach pool as objects with unique IDs
master_coach_pool = []
coach_counter = 1

for part in user_input.split(","):
    part = part.strip().lower()
    if not part:
        continue

    part = part.replace("*", "x").replace("-", "x").replace(" ", "")
    match = re.match(r"(\d+)x(\d+)", part)
    if not match:
        print(f"Skipping invalid entry: {part}")
        continue

    count = int(match.group(1))
    cap = int(match.group(2))

    for _ in range(count):
        master_coach_pool.append({
            "id": f"coach{coach_counter:03d}",
            "cap": cap
        })
        coach_counter += 1

# sort ASC by capacity for best-fit
master_coach_pool = sorted(master_coach_pool, key=lambda x: x["cap"])

if not master_coach_pool:
    raise ValueError("No valid coach capacity entered. Please rerun.")

print("\nCoach pool template (per pickup window):",
      [f"{c['id']}({c['cap']})" for c in master_coach_pool])

# ---------- helper: best-fit coach ----------
def pick_best_fit_coach(pool, pax_required):
    """
    Choose the SMALLEST coach that can fit pax_required.
    pool must be sorted ASC by 'cap'.
    Returns (coach_obj, new_pool).
    If none fits: (None, pool unchanged).
    """
    for i, coach in enumerate(pool):
        if coach["cap"] >= pax_required:
            new_pool = pool[:i] + pool[i+1:]
            return coach, new_pool
    return None, pool

# ---------- helper: neighbors within radius (bidirectional) ----------
def neighbors_within_radius(anchor, candidates_set, radius_km=1.5):
    fwd = dist_df[
        (dist_df["From_clean"] == anchor) &
        (dist_df["To_clean"].isin(candidates_set)) &
        (dist_df["distance_km"] <= radius_km)
    ][["To_clean", "distance_km"]].rename(columns={"To_clean": "hotel"})

    rev = dist_df[
        (dist_df["To_clean"] == anchor) &
        (dist_df["From_clean"].isin(candidates_set)) &
        (dist_df["distance_km"] <= radius_km)
    ][["From_clean", "distance_km"]].rename(columns={"From_clean": "hotel"})

    combo = pd.concat([fwd, rev], ignore_index=True).drop_duplicates(subset=["hotel"])
    combo = combo.sort_values(["distance_km", "hotel"], ascending=[True, True])
    return combo["hotel"].tolist()

def build_step2_batches_for_one_batch(group, coach_pool, radius_km=1.5):
    """
    Builds step2 batches for ONE (pickup_window, batch_id) group.
    Consumes from coach_pool and returns (updated_group, updated_coach_pool).
    """
    group = group.copy()

    # Aggregate pax at hotel-level (if a hotel appears multiple times in this batch)
    hotel_summary = (
        group.groupby("From_clean", dropna=False)
             .agg(
                 total_pack=("total_pack", "sum"),
                 dest=("To", lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0])
             )
             .reset_index()
    )

    packs = dict(zip(hotel_summary["From_clean"], hotel_summary["total_pack"]))
    dests = dict(zip(hotel_summary["From_clean"], hotel_summary["dest"]))

    unassigned = set(hotel_summary["From_clean"].tolist())

    hotel_to_sid = {}  # hotel -> step2_batch_id
    sid_to_coach = {}  # step2_batch_id -> coach_obj (or None)

    batch_prefix = group["batch_id"].iloc[0]
    step_counter = 1

    # RULE: DO NOT SPLIT if whole batch fits into one available coach
    total_pax = int(hotel_summary["total_pack"].sum())
    coach, new_pool = pick_best_fit_coach(coach_pool, total_pax)

    if coach is not None:
        sid = f"{batch_prefix}-1"
        for h in unassigned:
            hotel_to_sid[h] = sid

        sid_to_coach[sid] = coach
        coach_pool = new_pool

        group["step2_batch_id"] = group["From_clean"].map(hotel_to_sid)

        # NEW OUTPUT COLUMNS
        group["vehicle_id"] = group["step2_batch_id"].map(
            lambda x: sid_to_coach[x]["id"] if x in sid_to_coach and sid_to_coach[x] else None
        )
        group["vehicle_capacity"] = group["step2_batch_id"].map(
            lambda x: sid_to_coach[x]["cap"] if x in sid_to_coach and sid_to_coach[x] else None
        )
        group["vehicle_label"] = group.apply(
            lambda r: f"{r['vehicle_id']} ({int(r['vehicle_capacity'])} seater)"
            if pd.notna(r["vehicle_id"]) and pd.notna(r["vehicle_capacity"]) else None,
            axis=1
        )

        # Keep your old column name too (optional; helps compatibility)
        group["assigned_vehicle_capacity"] = group["vehicle_capacity"]
        return group, coach_pool

    # Otherwise, split greedily but ALWAYS check feasibility vs CURRENT pool
    while unassigned:
        anchor = max(unassigned, key=lambda h: packs.get(h, 0))
        current = [anchor]
        current_pax = int(packs[anchor])

        candidates = unassigned - {anchor}
        neigh_list = neighbors_within_radius(anchor, candidates, radius_km=radius_km)

        # add same-destination hotels after neighbors (reduces singleton risk)
        same_dest = [h for h in candidates if dests.get(h) == dests.get(anchor) and h not in neigh_list]
        same_dest = sorted(same_dest, key=lambda h: packs.get(h, 0), reverse=True)

        ordered_try = neigh_list + same_dest

        for h in ordered_try:
            if h not in unassigned:
                continue

            new_pax = current_pax + int(packs.get(h, 0))

            # feasibility check: is there ANY remaining coach that can fit new_pax?
            feasible = any(c["cap"] >= new_pax for c in coach_pool)

            if feasible:
                current.append(h)
                current_pax = new_pax

        # allocate best-fit coach (if available)
        coach, new_pool = pick_best_fit_coach(coach_pool, current_pax)
        if coach is not None:
            coach_pool = new_pool

        sid = f"{batch_prefix}-{step_counter}"
        step_counter += 1

        for h in current:
            hotel_to_sid[h] = sid
            unassigned.discard(h)

        sid_to_coach[sid] = coach  # could be None if no coach fits

    group["step2_batch_id"] = group["From_clean"].map(hotel_to_sid)

    # NEW OUTPUT COLUMNS
    group["vehicle_id"] = group["step2_batch_id"].map(
        lambda x: sid_to_coach[x]["id"] if x in sid_to_coach and sid_to_coach[x] else None
    )
    group["vehicle_capacity"] = group["step2_batch_id"].map(
        lambda x: sid_to_coach[x]["cap"] if x in sid_to_coach and sid_to_coach[x] else None
    )
    group["vehicle_label"] = group.apply(
        lambda r: f"{r['vehicle_id']} ({int(r['vehicle_capacity'])} seater)"
        if pd.notna(r["vehicle_id"]) and pd.notna(r["vehicle_capacity"]) else None,
        axis=1
    )

    # Keep your old column name too (optional; helps compatibility)
    group["assigned_vehicle_capacity"] = group["vehicle_capacity"]

    return group, coach_pool

# -------- APPLY STEP2 (true reuse per pickup_window) --------
# Initialize output columns
df["step2_batch_id"] = None
df["assigned_vehicle_capacity"] = None
df["vehicle_id"] = None
df["vehicle_capacity"] = None
df["vehicle_label"] = None

pickup_df = df[df["trip_type"].str.lower().str.contains("pickup", na=False)].copy()

def process_one_pickup_window(window_group):
    # IMPORTANT: reset pool ONCE per pickup_window; reuse happens within this window
    coach_pool = master_coach_pool.copy()
    outputs = []

    for batch_id, grp in window_group.groupby("batch_id", sort=True):
        grp2, coach_pool = build_step2_batches_for_one_batch(grp, coach_pool, radius_km=1.5)
        outputs.append(grp2)

    return pd.concat(outputs, ignore_index=False)

pickup_df = (
    pickup_df.groupby("pickup_window", group_keys=False, sort=True)
             .apply(process_one_pickup_window)
)

df.update(pickup_df)

# =====================================================
# FINAL OUTPUT
# =====================================================
final_df = (
    df.sort_values(["pickup_window", "batch_id", "step2_batch_id"])
      .reset_index(drop=True)
)

final_df.to_excel("88.xlsx", index=False)

print("\n🎯 FINAL FILE GENERATED → 88.xlsx")
print("✔️ Vehicles reused per pickup window (same coach ID reused inside window)")
print("✔️ Do-not-split rule uses best-fit coach from remaining pool")
print("✔️ Splitting uses feasibility vs remaining capacities (no max(pool) bug)")
print("✔️ Hotel pax aggregated inside each batch before splitting")
print("✔️ Added vehicle_id / vehicle_capacity / vehicle_label columns")
print("✔️ assigned_vehicle_capacity kept for compatibility\n")